In [127]:
import torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification

In [124]:
# MODEL_PATH = 'short-mamba/ettin-1b-drm-safety-classifier'
MODEL_PATH = '/data/user_data/jamesdin/models/ettin-1b-drm-safety-classifier'

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

# Load label mappings
id2label = model.config.id2label
label2id = model.config.label2id

print(f"Model loaded from {MODEL_PATH}")
print(f"Number of classes: {len(id2label)}")
print(f"Labels: {list(id2label.values())}\n")


Model loaded from /data/user_data/jamesdin/models/ettin-1b-drm-safety-classifier
Number of classes: 6
Labels: ['direct_harmful_content', 'external_reference', 'norm_violation_flag', 'other', 'safe_strategy_conversion', 'user_intent_inference']



In [131]:
model_config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)

In [138]:
model_config.num_labels

6

In [140]:
tokenizer.model_max_length


8192

GRPO w/ Process Supervision

In [145]:
import numpy as np
from typing import Optional


In [150]:
def compute_grpo_process_advantage(
    token_level_rewards: torch.Tensor,
    response_mask: torch.Tensor,
    step_mask: torch.Tensor,
    index: np.ndarray,
    epsilon: float = 1e-6,
    norm_adv_by_std_in_grpo: bool = True,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute advantage for GRPO with process (step / chunk) supervision.

    This implements the "GRPO with process rewards" case:
      1) Normalize all step-level rewards across the solutions of the same prompt.
      2) For each token, its advantage is the cumulative sum of normalized step rewards
         from all steps whose end index >= current token index (suffix sum).

    Args:
        token_level_rewards: `(torch.Tensor)`
            Shape: (bs, response_length).
            Token-level rewards. Only tokens that correspond to the end of a step/chunk
            need to have non-zero rewards (others can be 0).
        response_mask: `(torch.Tensor)`
            Shape: (bs, response_length). 1 for valid tokens, 0 for padding.
        step_mask: `(torch.Tensor)`
            Shape: (bs, response_length). 1 at the last token of each step/chunk, 0 otherwise.
        index: `(np.ndarray)`
            Index array for grouping (e.g., all samples that share the same prompt).
        epsilon: `(float)`
            Small value to avoid division by zero when normalizing by std.
        norm_adv_by_std_in_grpo: `(bool)`
            If True, divide by std as in original GRPO. If False, only mean-center
            (Dr.GRPO-style).
        config: `(Optional[AlgoConfig])`
            Algorithm configuration object (kept for API compatibility; not used here).

    Returns:
        advantages: `(torch.Tensor)`
            Shape: (bs, response_length).
        returns: `(torch.Tensor)`
            Shape: (bs, response_length). Same as `advantages` for GRPO-style MC returns.
    """
    # All tensors should have the same shape
    assert token_level_rewards.shape == response_mask.shape == step_mask.shape, (
        "token_level_rewards, response_mask, and step_mask must have the same shape"
    )

    # Ensure masks are float tensors (0/1) and on the same device/dtype
    response_mask = response_mask.to(token_level_rewards.dtype)
    step_mask = step_mask.to(token_level_rewards.dtype)

    from collections import defaultdict

    with torch.no_grad():
        bsz, seqlen = token_level_rewards.shape

        # Step rewards exist only at step boundaries.
        # This zeroes out non-step tokens and padded tokens.
        step_rewards_all = token_level_rewards * step_mask * response_mask

        # Collect step-level rewards per *group* (prompt) so we can do
        # group-wise normalization as in GRPO.
        id2_step_rewards: dict[int, list[torch.Tensor]] = defaultdict(list)
        id2_mean: dict[int, torch.Tensor] = {}
        id2_std: dict[int, torch.Tensor] = {}

        for i in range(bsz):
            grp_id = index[i]
            # Positions that are (1) step ends and (2) valid response tokens
            mask_i = (step_mask[i] > 0) & (response_mask[i] > 0)
            rewards_i = step_rewards_all[i][mask_i]

            if rewards_i.numel() == 0:
                raise ValueError(
                    f"No step rewards found for sample {i} (group id {grp_id}). "
                    "Make sure `step_mask` marks the end of each step/chunk."
                )

            for r in rewards_i:
                id2_step_rewards[grp_id].append(r)
        
        print('id2_step_rewards', id2_step_rewards)

        # Per-group mean/std over all step rewards R (Eq. 3).
        for grp_id, rewards in id2_step_rewards.items():
            if len(rewards) == 1:
                # Degenerate case: fall back to using the raw reward directly.
                r0 = rewards[0]
                id2_mean[grp_id] = torch.tensor(
                    0.0, device=r0.device, dtype=r0.dtype
                )
                id2_std[grp_id] = torch.tensor(
                    1.0, device=r0.device, dtype=r0.dtype
                )
            elif len(rewards) > 1:
                stacked = torch.stack(rewards)
                id2_mean[grp_id] = torch.mean(stacked)
                id2_std[grp_id] = torch.std(stacked)
            else:
                raise ValueError(f"No step reward in group id: {grp_id}")
        
        print('id2_mean', id2_mean)
        print('id2_std', id2_std)

        # Normalize step rewards and scatter them back into token positions.
        norm_step_rewards = torch.zeros_like(token_level_rewards)

        for i in range(bsz):
            grp_id = index[i]
            mean = id2_mean[grp_id]
            std = id2_std[grp_id]

            mask_i = (step_mask[i] > 0) & (response_mask[i] > 0)
            rewards_i = step_rewards_all[i][mask_i]

            if norm_adv_by_std_in_grpo:
                norm_i = (rewards_i - mean) / (std + epsilon)
            else:
                norm_i = rewards_i - mean

            norm_step_rewards[i][mask_i] = norm_i
        
        print('norm_step_rewards', norm_step_rewards)

        # Respect response_mask (no effect on already-masked positions, but keeps things explicit).
        norm_step_rewards = norm_step_rewards * response_mask

        # Compute token-level advantages as suffix sums over normalized step rewards (Eq. 4):
        #   A_{i,t} = sum_{index(j) >= t} \hat{r}_{index(j)}^i
        # Since norm_step_rewards is non-zero only at step ends, this exactly matches the paper.
        advantages = torch.flip(
            torch.cumsum(torch.flip(norm_step_rewards, dims=[-1]), dim=-1),
            dims=[-1],
        )
        advantages = advantages * response_mask

    # As in your outcome-based GRPO, we use the same tensor for advantages and returns.
    return advantages, advantages


In [151]:
bsz, seqlen = 1, 5

token_level_rewards = torch.tensor([[0.0, 1.0, 0.0, 0.0, 3.0]])  # (1, 5)
response_mask       = torch.ones_like(token_level_rewards)       # all valid tokens
step_mask           = torch.tensor([[0, 1, 0, 0, 1]], dtype=torch.float32)  # step ends at 1 and 4
index               = np.array([0])  # single prompt group

advantages, returns = compute_grpo_process_advantage(
    token_level_rewards=token_level_rewards,
    response_mask=response_mask,
    step_mask=step_mask,
    index=index,
    norm_adv_by_std_in_grpo=True,
)

print("token_level_rewards:\n", token_level_rewards)
print("step_mask:\n", step_mask)
print("advantages:\n", advantages)
print("returns:\n", returns)


id2_step_rewards defaultdict(<class 'list'>, {0: [tensor(1.), tensor(3.)]})
id2_mean {0: tensor(2.)}
id2_std {0: tensor(1.4142)}
norm_step_rewards tensor([[ 0.0000, -0.7071,  0.0000,  0.0000,  0.7071]])
token_level_rewards:
 tensor([[0., 1., 0., 0., 3.]])
step_mask:
 tensor([[0., 1., 0., 0., 1.]])
advantages:
 tensor([[0.0000, 0.0000, 0.7071, 0.7071, 0.7071]])
returns:
 tensor([[0.0000, 0.0000, 0.7071, 0.7071, 0.7071]])


In [152]:
bsz, seqlen = 2, 5

token_level_rewards = torch.tensor([
    [0.0,  1.0, 0.0, 0.0,  3.0],   # sample 0
    [0.0,  5.0, 0.0, 0.0, -1.0],   # sample 1
])
response_mask = torch.ones_like(token_level_rewards)

# Step ends at positions 1 and 4 again for both sequences
step_mask = torch.tensor([
    [0, 1, 0, 0, 1],
    [0, 1, 0, 0, 1],
], dtype=torch.float32)

index = np.array([0, 0])  # both samples belong to the same group

advantages, returns = compute_grpo_process_advantage(
    token_level_rewards=token_level_rewards,
    response_mask=response_mask,
    step_mask=step_mask,
    index=index,
    norm_adv_by_std_in_grpo=True,
)

print("token_level_rewards:\n", token_level_rewards)
print("step_mask:\n", step_mask)
print("advantages:\n", advantages)


id2_step_rewards defaultdict(<class 'list'>, {0: [tensor(1.), tensor(3.), tensor(5.), tensor(-1.)]})
id2_mean {0: tensor(2.)}
id2_std {0: tensor(2.5820)}
norm_step_rewards tensor([[ 0.0000, -0.3873,  0.0000,  0.0000,  0.3873],
        [ 0.0000,  1.1619,  0.0000,  0.0000, -1.1619]])
token_level_rewards:
 tensor([[ 0.,  1.,  0.,  0.,  3.],
        [ 0.,  5.,  0.,  0., -1.]])
step_mask:
 tensor([[0., 1., 0., 0., 1.],
        [0., 1., 0., 0., 1.]])
advantages:
 tensor([[ 0.0000,  0.0000,  0.3873,  0.3873,  0.3873],
        [ 0.0000,  0.0000, -1.1619, -1.1619, -1.1619]])


In [154]:
bsz, seqlen = 4, 5

token_level_rewards = torch.tensor([
    [0.0,  1.0, 0.0, 0.0,  3.0],   # group 0
    [0.0, 10.0, 0.0, 0.0, 10.0],   # group 0
    [0.0,  0.0, 2.0, 0.0,  5.0],   # group 1
    [1.0, 0.0, 0.0, 0.0, 7.0],   # group 1
])
response_mask = torch.ones_like(token_level_rewards)
step_mask = torch.tensor([
    [0, 1, 0, 0, 1],
    [0, 1, 0, 0, 1],
    [0, 0, 1, 0, 1],
    [1, 0, 0, 0, 1],
], dtype=torch.float32)

index = np.array([0, 0, 1,1])  # different prompt groups

advantages, returns = compute_grpo_process_advantage(
    token_level_rewards=token_level_rewards,
    response_mask=response_mask,
    step_mask=step_mask,
    index=index,
    norm_adv_by_std_in_grpo=True,
)

print("token_level_rewards:\n", token_level_rewards)
print("step_mask:\n", step_mask)
print("index:", index)
print("advantages:\n", advantages)


id2_step_rewards defaultdict(<class 'list'>, {0: [tensor(1.), tensor(3.), tensor(10.), tensor(10.)], 1: [tensor(2.), tensor(5.), tensor(1.), tensor(7.)]})
id2_mean {0: tensor(6.), 1: tensor(3.7500)}
id2_std {0: tensor(4.6904), 1: tensor(2.7538)}
norm_step_rewards tensor([[ 0.0000, -1.0660,  0.0000,  0.0000, -0.6396],
        [ 0.0000,  0.8528,  0.0000,  0.0000,  0.8528],
        [ 0.0000,  0.0000, -0.6355,  0.0000,  0.4539],
        [-0.9986,  0.0000,  0.0000,  0.0000,  1.1802]])
token_level_rewards:
 tensor([[ 0.,  1.,  0.,  0.,  3.],
        [ 0., 10.,  0.,  0., 10.],
        [ 0.,  0.,  2.,  0.,  5.],
        [ 1.,  0.,  0.,  0.,  7.]])
step_mask:
 tensor([[0., 1., 0., 0., 1.],
        [0., 1., 0., 0., 1.],
        [0., 0., 1., 0., 1.],
        [1., 0., 0., 0., 1.]])
index: [0 0 1 1]
advantages:
 tensor([[-1.7056, -1.7056, -0.6396, -0.6396, -0.6396],
        [ 1.7056,  1.7056,  0.8528,  0.8528,  0.8528],
        [-0.1816, -0.1816, -0.1816,  0.4539,  0.4539],
        [ 0.1816,  1.180